In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis_code').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis_code' / '03_dtw_phenotypes'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
from matplotlib.lines import Line2D
from pathlib import Path
from shapely.geometry import LineString


# =========================================================
# 0. 全局参数
# =========================================================

# ---------- 圆形城市点数据 ----------
city_cluster_path = Path(
    str(EXTERNAL_DATA_ROOT / "gis" / "city_cluster.shp")
)

# ---------- 城市名称字段 ----------
CITY_NAME_FIELD = "city_name"

# ---------- 棱形覆盖点数据 ----------
overlay_gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
overlay_layer = "city_cluster_ExportFeatures"

# ---------- 美国边界数据 ----------
gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
outline_layer = "Outline"

states_shp = str(EXTERNAL_DATA_ROOT / "gis" / "US_states.shp")

# ---------- 输出文件夹 ----------
out_dir = Path(
    str(MODULE_DIR / "output")
)
out_dir.mkdir(parents=True, exist_ok=True)

output_name = "city_cluster_spatial_layout_label_greedy_legend"

# ---------- 目标投影 ----------
TARGET_CRS = "EPSG:5070"

# ---------- 图形尺寸 ----------
FIG_WIDTH = 10.5
FIG_HEIGHT_RATIO = 0.68
FIG_HEIGHT = FIG_WIDTH * FIG_HEIGHT_RATIO
DPI = 600

# ---------- 字体 ----------
FONT_FAMILY = "Times New Roman"
plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# =========================================================
# 1. 点图层样式
# =========================================================

# cluster = 0 灰色方块，也添加白边
BASE_POINT_STYLES = {
    0: {
        "marker": "s",
        "facecolor": "#BFBFBF",
        "edgecolor": "white",
        "size": 85,
        "linewidth": 0.7,
        "label": "Cluster 0"
    },
    1: {
        "marker": "o",
        "facecolor": "#A92A24",
        "edgecolor": "white",
        "size": 105,
        "linewidth": 0.6,
        "label": "Cluster 1"
    },
    2: {
        "marker": "o",
        "facecolor": "#F0D27A",
        "edgecolor": "white",
        "size": 105,
        "linewidth": 0.6,
        "label": "Cluster 2"
    },
    3: {
        "marker": "o",
        "facecolor": "#AFCBE6",
        "edgecolor": "white",
        "size": 105,
        "linewidth": 0.6,
        "label": "Cluster 3"
    },
    4: {
        "marker": "o",
        "facecolor": "#4B88C7",
        "edgecolor": "white",
        "size": 105,
        "linewidth": 0.6,
        "label": "Cluster 4"
    },
}

# 棱形覆盖点，作为 outlier 样本，位于最上层
OVERLAY_POINT_STYLES = {
    0: {
        "marker": "D",
        "facecolor": "#BFBFBF",
        "edgecolor": "white",
        "size": 150,
        "linewidth": 0.8
    },
    1: {
        "marker": "D",
        "facecolor": "#CF2E25",
        "edgecolor": "white",
        "size": 150,
        "linewidth": 0.8
    },
    2: {
        "marker": "D",
        "facecolor": "#F3E34F",
        "edgecolor": "white",
        "size": 150,
        "linewidth": 0.8
    },
    3: {
        "marker": "D",
        "facecolor": "#BFD4E6",
        "edgecolor": "white",
        "size": 150,
        "linewidth": 0.8
    },
    4: {
        "marker": "D",
        "facecolor": "#5B8ED8",
        "edgecolor": "white",
        "size": 150,
        "linewidth": 0.8
    },
}


# =========================================================
# 2. 城市标注参数
# =========================================================

SHOW_CITY_LABELS = True

# None 表示标注所有城市
# 如果图面太密，可以改为 [1, 2, 3, 4]，不标注 cluster=0
LABEL_CLUSTER_VALUES = None

CITY_LABEL_SIZE = 7.2
CITY_LABEL_COLOR = "#8A8A8A"

CITY_LABEL_BBOX = False
CITY_LABEL_BBOX_ALPHA = 0.65

SKIP_OVERLAPPED_LABELS = True
KEEP_LABELS_INSIDE_AXES = True

# 点和文字之间的避让半径，单位为屏幕像素
# 增大该值可减少文字压住圆点/菱形的概率
BASE_POINT_OBSTACLE_RADIUS_PX = 13
OVERLAY_POINT_OBSTACLE_RADIUS_PX = 17

# 文字框放大系数，越大则文字之间留白越多
LABEL_BBOX_EXPAND_X = 1.08
LABEL_BBOX_EXPAND_Y = 1.25

# 候选偏移距离，单位为米，EPSG:5070
# 已增加更远的候选位置，尽量避免覆盖点和图形
LABEL_CANDIDATE_OFFSETS = [
    (26000, 18000, "left", "center"),
    (26000, -18000, "left", "center"),
    (-26000, 18000, "right", "center"),
    (-26000, -18000, "right", "center"),

    (38000, 0, "left", "center"),
    (-38000, 0, "right", "center"),
    (0, 36000, "center", "bottom"),
    (0, -36000, "center", "top"),

    (52000, 30000, "left", "center"),
    (52000, -30000, "left", "center"),
    (-52000, 30000, "right", "center"),
    (-52000, -30000, "right", "center"),

    (70000, 0, "left", "center"),
    (-70000, 0, "right", "center"),
    (0, 62000, "center", "bottom"),
    (0, -62000, "center", "top"),

    (90000, 45000, "left", "center"),
    (90000, -45000, "left", "center"),
    (-90000, 45000, "right", "center"),
    (-90000, -45000, "right", "center"),
]


# =========================================================
# 3. 边界、经纬网、图框参数
# =========================================================

OUTLINE_COLOR = "#666666"
OUTLINE_WIDTH = 1.2

STATE_LINE_COLOR = "#8A8A8A"
STATE_LINE_WIDTH = 0.50

GRID_LONS = [-120, -110, -100, -90, -80]
GRID_LATS = [20, 30, 40, 50]
LAT_LABELS = [30, 40, 50]

GRID_EXTEND_LON_MIN = -132
GRID_EXTEND_LON_MAX = -58
GRID_EXTEND_LAT_MIN = 12
GRID_EXTEND_LAT_MAX = 60

GRID_COLOR = "#BFBFBF"
GRID_WIDTH = 0.45
GRID_ALPHA = 0.90
GRID_STYLE = "-"

ADD_BOTTOM_LABEL_AXIS_LINE = True
BOTTOM_AXIS_LINE_WIDTH = 0.8
BOTTOM_AXIS_LINE_COLOR = "black"
BOTTOM_TICK_LENGTH_RATIO = 0.018
BOTTOM_LABEL_OFFSET_RATIO = 0.025

ADD_LEFT_LABEL_AXIS_LINE = True
LEFT_AXIS_LINE_WIDTH = 0.8
LEFT_AXIS_LINE_COLOR = "black"
LEFT_TICK_LENGTH_RATIO = 0.018
LEFT_LABEL_OFFSET_RATIO = 0.030

SPINE_WIDTH = 0.8

TICK_SIZE = 10
SCALE_TEXT_SIZE = 9

SHOW_SCALEBAR = True
SCALE_LENGTH_M = 100_000
SCALE_HEIGHT_M = 90_000
SCALE_X_FRAC = 0.070
SCALE_Y_FRAC = 0.070
SCALE_TEXT_OFFSET_M = 20_000

PAD_X_RATIO = 0.015
PAD_Y_RATIO = 0.020

DATA_FRAME_ASPECT_RATIO = 0.70
FIX_WIDTH_ADJUST_HEIGHT = True

# ---------- 图例 ----------
SHOW_LEGEND = True
LEGEND_FONT_SIZE = 9
LEGEND_MARKER_SIZE = 8
LEGEND_LOC = "lower right"
LEGEND_BBOX = (0.985, 0.075)
LEGEND_FRAME = True
LEGEND_FRAME_ALPHA = 0.95
LEGEND_EDGE_COLOR = "#888888"

SAVE_PNG = True
SAVE_SVG = True


# =========================================================
# 4. 辅助函数
# =========================================================

def make_lonlat_line(lon=None, lat=None, n=1200):
    if lon is not None:
        lats = np.linspace(GRID_EXTEND_LAT_MIN, GRID_EXTEND_LAT_MAX, n)
        coords = [(lon, y) for y in lats]
    elif lat is not None:
        lons = np.linspace(GRID_EXTEND_LON_MIN, GRID_EXTEND_LON_MAX, n)
        coords = [(x, lat) for x in lons]
    else:
        raise ValueError("lon 和 lat 至少需要指定一个")

    return gpd.GeoDataFrame(
        geometry=[LineString(coords)],
        crs="EPSG:4326"
    )


def _interp_x_at_y(x, y, y0, xlim):
    xs = []

    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]

        if (y1 - y0) * (y2 - y0) <= 0 and y1 != y2:
            t = (y0 - y1) / (y2 - y1)
            xi = x1 + t * (x2 - x1)

            if xlim[0] <= xi <= xlim[1]:
                xs.append(xi)

    if len(xs) == 0:
        return None

    x_center = (xlim[0] + xlim[1]) / 2
    return min(xs, key=lambda v: abs(v - x_center))


def _interp_y_at_x(x, y, x0, ylim):
    ys = []

    for i in range(len(x) - 1):
        x1, x2 = x[i], x[i + 1]
        y1, y2 = y[i], y[i + 1]

        if (x1 - x0) * (x2 - x0) <= 0 and x1 != x2:
            t = (x0 - x1) / (x2 - x1)
            yi = y1 + t * (y2 - y1)

            if ylim[0] <= yi <= ylim[1]:
                ys.append(yi)

    if len(ys) == 0:
        return None

    y_center = (ylim[0] + ylim[1]) / 2
    return min(ys, key=lambda v: abs(v - y_center))


def add_graticules(ax, target_crs):
    ax.set_xticks([])
    ax.set_yticks([])

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    lon_lines = {}
    lat_lines = {}

    for lon in GRID_LONS:
        line = make_lonlat_line(lon=lon).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lon_lines[lon] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    for lat in GRID_LATS:
        line = make_lonlat_line(lat=lat).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lat_lines[lat] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    y_axis = ylim[0]
    bottom_tick_len = y_range * BOTTOM_TICK_LENGTH_RATIO
    bottom_label_offset = y_range * BOTTOM_LABEL_OFFSET_RATIO

    if ADD_BOTTOM_LABEL_AXIS_LINE:
        ax.plot(
            [xlim[0], xlim[1]],
            [y_axis, y_axis],
            color=BOTTOM_AXIS_LINE_COLOR,
            linewidth=BOTTOM_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lon in GRID_LONS:
        x_line, y_line = lon_lines[lon]
        x_label = _interp_x_at_y(x_line, y_line, y_axis, xlim)

        if x_label is None:
            continue

        if ADD_BOTTOM_LABEL_AXIS_LINE:
            ax.plot(
                [x_label, x_label],
                [y_axis, y_axis - bottom_tick_len],
                color=BOTTOM_AXIS_LINE_COLOR,
                linewidth=BOTTOM_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_label,
            y_axis - bottom_label_offset,
            f"{abs(lon):.0f}°W",
            ha="center",
            va="top",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )

    x_axis = xlim[0]
    left_tick_len = x_range * LEFT_TICK_LENGTH_RATIO
    left_label_offset = x_range * LEFT_LABEL_OFFSET_RATIO

    if ADD_LEFT_LABEL_AXIS_LINE:
        ax.plot(
            [x_axis, x_axis],
            [ylim[0], ylim[1]],
            color=LEFT_AXIS_LINE_COLOR,
            linewidth=LEFT_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lat in LAT_LABELS:
        if lat not in lat_lines:
            continue

        x_line, y_line = lat_lines[lat]
        y_label = _interp_y_at_x(x_line, y_line, x_axis, ylim)

        if y_label is None:
            continue

        if ADD_LEFT_LABEL_AXIS_LINE:
            ax.plot(
                [x_axis, x_axis - left_tick_len],
                [y_label, y_label],
                color=LEFT_AXIS_LINE_COLOR,
                linewidth=LEFT_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_axis - left_label_offset,
            y_label,
            f"{lat:.0f}°N",
            ha="right",
            va="center",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )


def add_scale_bar(ax):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    x0 = xlim[0] + SCALE_X_FRAC * x_range
    y0 = ylim[0] + SCALE_Y_FRAC * y_range

    rect = Rectangle(
        (x0, y0),
        SCALE_LENGTH_M,
        SCALE_HEIGHT_M,
        facecolor="black",
        edgecolor="black",
        linewidth=0,
        zorder=30
    )
    ax.add_patch(rect)

    ax.text(
        x0 + SCALE_LENGTH_M / 2,
        y0 - SCALE_TEXT_OFFSET_M,
        "100 km",
        ha="center",
        va="top",
        fontsize=SCALE_TEXT_SIZE,
        color="black",
        zorder=30
    )


def get_display_xlim_ylim(outline_5070):
    xmin, ymin, xmax, ymax = outline_5070.total_bounds

    x_center = (xmin + xmax) / 2
    y_center = (ymin + ymax) / 2

    raw_width = xmax - xmin
    raw_height = ymax - ymin

    width_with_pad = raw_width * (1 + 2 * PAD_X_RATIO)
    height_with_pad = raw_height * (1 + 2 * PAD_Y_RATIO)

    if FIX_WIDTH_ADJUST_HEIGHT:
        target_width = width_with_pad
        target_height = target_width * DATA_FRAME_ASPECT_RATIO
        target_height = max(target_height, height_with_pad)
    else:
        target_height = height_with_pad
        target_width = target_height / DATA_FRAME_ASPECT_RATIO
        target_width = max(target_width, width_with_pad)

    xlim = (
        x_center - target_width / 2,
        x_center + target_width / 2
    )

    ylim = (
        y_center - target_height / 2,
        y_center + target_height / 2
    )

    return xlim, ylim


def make_point_obstacles(ax, gdf, radius_px):
    """
    将点位转换为屏幕坐标下的 bbox 障碍物，避免城市名压住圆点/菱形。
    """
    obstacles = []

    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue

        px, py = ax.transData.transform((geom.x, geom.y))

        obstacles.append(
            Bbox.from_extents(
                px - radius_px,
                py - radius_px,
                px + radius_px,
                py + radius_px
            )
        )

    return obstacles


def bbox_inside_axes(bbox, ax_bbox):
    return (
        bbox.x0 >= ax_bbox.x0 and
        bbox.x1 <= ax_bbox.x1 and
        bbox.y0 >= ax_bbox.y0 and
        bbox.y1 <= ax_bbox.y1
    )


def add_city_labels_greedy(ax, city_gdf, name_field, base_point_gdf, overlay_point_gdf):
    """
    不依赖 adjustText / scipy 的城市名避让函数。
    增强点：
    1. 圆形点和菱形点分别作为障碍物；
    2. 城市名不会压住圆点或菱形；
    3. 若所有候选位置均重叠，则跳过该城市名。
    """
    if name_field not in city_gdf.columns:
        raise ValueError(f"城市点数据中未找到字段：{name_field}")

    if LABEL_CLUSTER_VALUES is not None:
        label_gdf = city_gdf[
            city_gdf["cluster"].isin(LABEL_CLUSTER_VALUES)
        ].copy()
    else:
        label_gdf = city_gdf.copy()

    label_gdf = label_gdf[
        label_gdf.geometry.notnull() &
        (~label_gdf.geometry.is_empty)
    ].copy()

    label_gdf["_label_priority"] = np.where(label_gdf["cluster"] == 0, 1, 0)
    label_gdf = label_gdf.sort_values(
        by=["_label_priority", "cluster"],
        ascending=[True, True]
    )

    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ax_bbox = ax.get_window_extent(renderer)

    placed_bboxes = []

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=base_point_gdf,
            radius_px=BASE_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=overlay_point_gdf,
            radius_px=OVERLAY_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_count = 0
    skipped_count = 0

    for _, row in label_gdf.iterrows():
        name = row[name_field]

        if name is None or str(name).strip() == "":
            continue

        x0 = row.geometry.x
        y0 = row.geometry.y

        placed = False

        for dx, dy, ha, va in LABEL_CANDIDATE_OFFSETS:

            if CITY_LABEL_BBOX:
                bbox_style = dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=CITY_LABEL_BBOX_ALPHA,
                    pad=0.15
                )
            else:
                bbox_style = None

            txt = ax.text(
                x0 + dx,
                y0 + dy,
                str(name),
                fontsize=CITY_LABEL_SIZE,
                color=CITY_LABEL_COLOR,
                ha=ha,
                va=va,
                bbox=bbox_style,
                zorder=18
            )

            fig.canvas.draw()
            bbox = txt.get_window_extent(renderer).expanded(
                LABEL_BBOX_EXPAND_X,
                LABEL_BBOX_EXPAND_Y
            )

            if KEEP_LABELS_INSIDE_AXES and not bbox_inside_axes(bbox, ax_bbox):
                txt.remove()
                continue

            has_overlap = any(
                bbox.overlaps(old_bbox)
                for old_bbox in placed_bboxes
            )

            if has_overlap:
                txt.remove()
                continue

            placed_bboxes.append(bbox)
            placed = True
            placed_count += 1
            break

        if not placed:
            skipped_count += 1

            if not SKIP_OVERLAPPED_LABELS:
                dx, dy, ha, va = LABEL_CANDIDATE_OFFSETS[0]
                txt = ax.text(
                    x0 + dx,
                    y0 + dy,
                    str(name),
                    fontsize=CITY_LABEL_SIZE,
                    color=CITY_LABEL_COLOR,
                    ha=ha,
                    va=va,
                    zorder=18
                )
                fig.canvas.draw()
                bbox = txt.get_window_extent(renderer).expanded(
                    LABEL_BBOX_EXPAND_X,
                    LABEL_BBOX_EXPAND_Y
                )
                placed_bboxes.append(bbox)

    print(f"城市名标注完成：成功 {placed_count} 个，跳过 {skipped_count} 个。")


def add_cluster_legend(ax):
    """
    添加 cluster 与 outlier 图例。
    cluster 0–4 使用圆形/方形符号；
    outlier 使用菱形符号。
    """
    handles = []

    for cls in [0, 1, 2, 3, 4]:
        style = BASE_POINT_STYLES[cls]

        handle = Line2D(
            [0],
            [0],
            marker=style["marker"],
            linestyle="None",
            label=style["label"],
            markerfacecolor=style["facecolor"],
            markeredgecolor=style["edgecolor"],
            markeredgewidth=style["linewidth"],
            markersize=LEGEND_MARKER_SIZE
        )
        handles.append(handle)

    outlier_handle = Line2D(
        [0],
        [0],
        marker="D",
        linestyle="None",
        label="Outlier",
        markerfacecolor="#FFFFFF",
        markeredgecolor="#555555",
        markeredgewidth=0.9,
        markersize=LEGEND_MARKER_SIZE
    )

    handles.append(outlier_handle)

    leg = ax.legend(
        handles=handles,
        loc=LEGEND_LOC,
        bbox_to_anchor=LEGEND_BBOX,
        fontsize=LEGEND_FONT_SIZE,
        frameon=LEGEND_FRAME,
        framealpha=LEGEND_FRAME_ALPHA,
        edgecolor=LEGEND_EDGE_COLOR,
        borderpad=0.6,
        handletextpad=0.6,
        labelspacing=0.45
    )

    leg.set_zorder(40)


# =========================================================
# 5. 读取数据
# =========================================================

outline_gdf = gpd.read_file(gdb_path, layer=outline_layer)
states_gdf = gpd.read_file(states_shp)

outline_wgs84 = outline_gdf.to_crs(epsg=4326)
states_wgs84 = states_gdf.to_crs(epsg=4326)

outline_5070 = outline_wgs84.to_crs(TARGET_CRS)
states_5070 = states_wgs84.to_crs(TARGET_CRS)

xlim, ylim = get_display_xlim_ylim(outline_5070)

city_gdf = gpd.read_file(city_cluster_path)
city_gdf = city_gdf.to_crs(TARGET_CRS)

overlay_gdf = gpd.read_file(overlay_gdb_path, layer=overlay_layer)
overlay_gdf = overlay_gdf.to_crs(TARGET_CRS)

if "cluster" not in city_gdf.columns:
    raise ValueError("圆形城市点数据中未找到字段：cluster")

if "cluster" not in overlay_gdf.columns:
    raise ValueError("棱形覆盖点数据中未找到字段：cluster")

if CITY_NAME_FIELD not in city_gdf.columns:
    raise ValueError(f"圆形城市点数据中未找到城市名字段：{CITY_NAME_FIELD}")

city_gdf["cluster"] = pd.to_numeric(
    city_gdf["cluster"],
    errors="coerce"
).astype("Int64")

overlay_gdf["cluster"] = pd.to_numeric(
    overlay_gdf["cluster"],
    errors="coerce"
).astype("Int64")

city_gdf = city_gdf.dropna(subset=["cluster"]).copy()
overlay_gdf = overlay_gdf.dropna(subset=["cluster"]).copy()

city_gdf["cluster"] = city_gdf["cluster"].astype(int)
overlay_gdf["cluster"] = overlay_gdf["cluster"].astype(int)


# =========================================================
# 6. 绘图
# =========================================================

fig, ax = plt.subplots(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    dpi=DPI,
    facecolor="white"
)

ax.set_facecolor("white")
ax.set_xlim(xlim)
ax.set_ylim(ylim)

add_graticules(ax, TARGET_CRS)

states_5070.boundary.plot(
    ax=ax,
    color=STATE_LINE_COLOR,
    linewidth=STATE_LINE_WIDTH,
    zorder=3
)

outline_5070.boundary.plot(
    ax=ax,
    color=OUTLINE_COLOR,
    linewidth=OUTLINE_WIDTH,
    zorder=4
)

# ---------- 圆形城市点 ----------
for cls, style in BASE_POINT_STYLES.items():
    sub = city_gdf[city_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=10
    )

# ---------- 城市名称，避让圆点和菱形 ----------
if SHOW_CITY_LABELS:
    add_city_labels_greedy(
        ax=ax,
        city_gdf=city_gdf,
        name_field=CITY_NAME_FIELD,
        base_point_gdf=city_gdf,
        overlay_point_gdf=overlay_gdf
    )

# ---------- 棱形覆盖点，最上层 ----------
for cls, style in OVERLAY_POINT_STYLES.items():
    sub = overlay_gdf[overlay_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=25
    )

if SHOW_SCALEBAR:
    add_scale_bar(ax)

if SHOW_LEGEND:
    add_cluster_legend(ax)

for spine in ax.spines.values():
    spine.set_linewidth(SPINE_WIDTH)
    spine.set_color("black")


# =========================================================
# 7. 保存
# =========================================================

plt.subplots_adjust(
    left=0.075,
    right=0.995,
    bottom=0.08,
    top=0.99
)

if SAVE_PNG:
    out_png = out_dir / f"{output_name}.png"
    plt.savefig(
        out_png,
        dpi=DPI,
        facecolor="white"
    )
    print(f"PNG 已保存：{out_png}")

if SAVE_SVG:
    out_svg = out_dir / f"{output_name}.svg"
    plt.savefig(
        out_svg,
        format="svg",
        facecolor="white"
    )
    print(f"SVG 已保存：{out_svg}")

plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
from matplotlib.lines import Line2D
from pathlib import Path
from shapely.geometry import LineString


# =========================================================
# 0. 全局参数
# =========================================================

# ---------- 圆形城市点数据 ----------
city_cluster_path = Path(
    str(EXTERNAL_DATA_ROOT / "gis" / "city_cluster.shp")
)

# ---------- 城市名称字段 ----------
CITY_NAME_FIELD = "city_name"

# ---------- 棱形覆盖点数据 ----------
overlay_gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
overlay_layer = "city_cluster_ExportFeatures"

# ---------- 美国边界数据 ----------
gdb_path = str(EXTERNAL_DATA_ROOT / "gis" / "MyProject1.gdb")
outline_layer = "Outline"

states_shp = str(EXTERNAL_DATA_ROOT / "gis" / "US_states.shp")

# ---------- 输出文件夹 ----------
out_dir = Path(
    str(MODULE_DIR / "output")
)
out_dir.mkdir(parents=True, exist_ok=True)

output_name = "city_cluster_spatial_layout_label_bigger_citynames_no_overlap_smaller_us_10km_above_legend"

# ---------- 目标投影 ----------
TARGET_CRS = "EPSG:5070"

# ---------- 图形尺寸 ----------
FIG_WIDTH = 10.5
FIG_HEIGHT_RATIO = 0.68
FIG_HEIGHT = FIG_WIDTH * FIG_HEIGHT_RATIO
DPI = 600

# ---------- 字体 ----------
FONT_FAMILY = "Times New Roman"
plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


# =========================================================
# 1. 点图层样式
# =========================================================

# ---------- cluster 颜色 ----------
COLOR_C1 = "#B23A3A"      # red
COLOR_C2 = "#E69F00"      # orange
COLOR_C3 = "#67A9CF"      # light blue
COLOR_C4 = "#1F4E8C"      # dark blue
COLOR_NO_HW = "#9E9E9E"   # gray

# ---------- 美国底图颜色 ----------
USA_FACE_COLOR = "#F0F0F0"
USA_EDGE_COLOR = "#B8B8B8"

# ---------- 空间点大小控制 ----------
NO_HEATWAVE_POINT_SIZE = 120
CLUSTER_POINT_SIZE = 150
OUTLIER_POINT_SIZE = 230

BASE_POINT_STYLES = {
    0: {
        "marker": "s",
        "facecolor": COLOR_NO_HW,
        "edgecolor": "white",
        "size": NO_HEATWAVE_POINT_SIZE,
        "linewidth": 0.8,
        "label": "No composite heatwave (n=12)"
    },
    1: {
        "marker": "o",
        "facecolor": COLOR_C1,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C1 (n=18)"
    },
    2: {
        "marker": "o",
        "facecolor": COLOR_C2,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C2 (n=14)"
    },
    3: {
        "marker": "o",
        "facecolor": COLOR_C3,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C3 (n=11)"
    },
    4: {
        "marker": "o",
        "facecolor": COLOR_C4,
        "edgecolor": "white",
        "size": CLUSTER_POINT_SIZE,
        "linewidth": 0.8,
        "label": "C4 (n=20)"
    },
}

# ---------- 棱形覆盖点，作为 outlier 样本，位于最上层 ----------
OVERLAY_POINT_STYLES = {
    0: {
        "marker": "D",
        "facecolor": COLOR_NO_HW,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    1: {
        "marker": "D",
        "facecolor": COLOR_C1,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    2: {
        "marker": "D",
        "facecolor": COLOR_C2,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    3: {
        "marker": "D",
        "facecolor": COLOR_C3,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
    4: {
        "marker": "D",
        "facecolor": COLOR_C4,
        "edgecolor": "black",
        "size": OUTLIER_POINT_SIZE,
        "linewidth": 1.2
    },
}


# =========================================================
# 2. 城市标注参数
# =========================================================

SHOW_CITY_LABELS = True

# None 表示标注所有城市
# 若图面太密，可以改为 [1, 2, 3, 4]，不标注 cluster=0
LABEL_CLUSTER_VALUES = None

# ---------- 城市名称字体 ----------
CITY_LABEL_SIZE = 8.5
CITY_LABEL_COLOR = "#8A8A8A"

CITY_LABEL_BBOX = False
CITY_LABEL_BBOX_ALPHA = 0.65

SKIP_OVERLAPPED_LABELS = True
KEEP_LABELS_INSIDE_AXES = True

# ---------- 点位障碍半径，避免文字压住圆点或菱形点 ----------
BASE_POINT_OBSTACLE_RADIUS_PX = 22
OVERLAY_POINT_OBSTACLE_RADIUS_PX = 30

# ---------- 文字框膨胀系数，减少城市名之间重叠 ----------
LABEL_BBOX_EXPAND_X = 1.18
LABEL_BBOX_EXPAND_Y = 1.35

# ---------- 城市名与点位的距离控制 ----------
# 数值越大，城市名离散点越远
LABEL_DISTANCE_SCALE = 1.45

LABEL_CANDIDATE_OFFSETS_BASE = [
    (26000, 18000, "left", "center"),
    (26000, -18000, "left", "center"),
    (-26000, 18000, "right", "center"),
    (-26000, -18000, "right", "center"),

    (38000, 0, "left", "center"),
    (-38000, 0, "right", "center"),
    (0, 36000, "center", "bottom"),
    (0, -36000, "center", "top"),

    (52000, 30000, "left", "center"),
    (52000, -30000, "left", "center"),
    (-52000, 30000, "right", "center"),
    (-52000, -30000, "right", "center"),

    (70000, 0, "left", "center"),
    (-70000, 0, "right", "center"),
    (0, 62000, "center", "bottom"),
    (0, -62000, "center", "top"),

    (90000, 45000, "left", "center"),
    (90000, -45000, "left", "center"),
    (-90000, 45000, "right", "center"),
    (-90000, -45000, "right", "center"),

    # 更远候选位置，适合东部密集城市群
    (115000, 60000, "left", "center"),
    (115000, -60000, "left", "center"),
    (-115000, 60000, "right", "center"),
    (-115000, -60000, "right", "center"),

    (140000, 0, "left", "center"),
    (-140000, 0, "right", "center"),
    (0, 100000, "center", "bottom"),
    (0, -100000, "center", "top"),
]

LABEL_CANDIDATE_OFFSETS = [
    (
        dx * LABEL_DISTANCE_SCALE,
        dy * LABEL_DISTANCE_SCALE,
        ha,
        va
    )
    for dx, dy, ha, va in LABEL_CANDIDATE_OFFSETS_BASE
]


# =========================================================
# 3. 边界、经纬网、图框参数
# =========================================================

OUTLINE_COLOR = "#9A9A9A"
OUTLINE_WIDTH = 0.85

STATE_LINE_COLOR = "#B8B8B8"
STATE_LINE_WIDTH = 0.55

GRID_LONS = [-120, -110, -100, -90, -80]
GRID_LATS = [20, 30, 40, 50]
LAT_LABELS = [30, 40, 50]

GRID_EXTEND_LON_MIN = -132
GRID_EXTEND_LON_MAX = -58
GRID_EXTEND_LAT_MIN = 12
GRID_EXTEND_LAT_MAX = 60

GRID_COLOR = "#D0D0D0"
GRID_WIDTH = 0.45
GRID_ALPHA = 0.90
GRID_STYLE = "-"

ADD_BOTTOM_LABEL_AXIS_LINE = True
BOTTOM_AXIS_LINE_WIDTH = 0.8
BOTTOM_AXIS_LINE_COLOR = "black"
BOTTOM_TICK_LENGTH_RATIO = 0.018
BOTTOM_LABEL_OFFSET_RATIO = 0.025

ADD_LEFT_LABEL_AXIS_LINE = True
LEFT_AXIS_LINE_WIDTH = 0.8
LEFT_AXIS_LINE_COLOR = "black"
LEFT_TICK_LENGTH_RATIO = 0.018
LEFT_LABEL_OFFSET_RATIO = 0.030

SPINE_WIDTH = 0.8

TICK_SIZE = 10
SCALE_TEXT_SIZE = 9

# =========================================================
# 比例尺参数：10 km，并放在分类 legend 上方
# =========================================================
SHOW_SCALEBAR = True
SCALE_LENGTH_M = 10_000
SCALE_HEIGHT_M = 18_000
SCALE_X_FRAC = 0.070
SCALE_Y_FRAC = 0.245
SCALE_TEXT_OFFSET_M = 8_000
SCALE_LABEL_TEXT = "10 km"

# ---------- 基础显示范围边距 ----------
PAD_X_RATIO = 0.015
PAD_Y_RATIO = 0.020

# ---------- 数据框高宽比控制 ----------
DATA_FRAME_ASPECT_RATIO = 0.70
FIX_WIDTH_ADJUST_HEIGHT = True

# =========================================================
# 美国地图在图中的比例控制
# =========================================================
# 数值越大，美国地图在图中越小
# 推荐范围：1.08–1.30
MAP_SHRINK_FACTOR = 1.18

# 可选：单独控制横向和纵向缩放
MAP_SHRINK_FACTOR_X = 1.00
MAP_SHRINK_FACTOR_Y = 1.00

# 可选：地图中心微调，单位为显示范围比例
# 正值：向右 / 向上移动；负值：向左 / 向下移动
MAP_CENTER_SHIFT_X_RATIO = 0.00
MAP_CENTER_SHIFT_Y_RATIO = 0.00


# ---------- 图例 ----------
SHOW_LEGEND = True
LEGEND_FONT_SIZE = 10
LEGEND_MARKER_SIZE = 8.5

# 分类 legend 放在左下角，比例尺位于它上方
LEGEND_LOC = "lower left"
LEGEND_BBOX = (0.02, 0.02)

LEGEND_FRAME = True
LEGEND_FRAME_ALPHA = 0.92
LEGEND_EDGE_COLOR = "#D0D0D0"

SAVE_PNG = True
SAVE_SVG = True


# =========================================================
# 4. 辅助函数
# =========================================================

def make_lonlat_line(lon=None, lat=None, n=1200):
    if lon is not None:
        lats = np.linspace(GRID_EXTEND_LAT_MIN, GRID_EXTEND_LAT_MAX, n)
        coords = [(lon, y) for y in lats]
    elif lat is not None:
        lons = np.linspace(GRID_EXTEND_LON_MIN, GRID_EXTEND_LON_MAX, n)
        coords = [(x, lat) for x in lons]
    else:
        raise ValueError("lon 和 lat 至少需要指定一个")

    return gpd.GeoDataFrame(
        geometry=[LineString(coords)],
        crs="EPSG:4326"
    )


def _interp_x_at_y(x, y, y0, xlim):
    xs = []

    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]

        if (y1 - y0) * (y2 - y0) <= 0 and y1 != y2:
            t = (y0 - y1) / (y2 - y1)
            xi = x1 + t * (x2 - x1)

            if xlim[0] <= xi <= xlim[1]:
                xs.append(xi)

    if len(xs) == 0:
        return None

    x_center = (xlim[0] + xlim[1]) / 2
    return min(xs, key=lambda v: abs(v - x_center))


def _interp_y_at_x(x, y, x0, ylim):
    ys = []

    for i in range(len(x) - 1):
        x1, x2 = x[i], x[i + 1]
        y1, y2 = y[i], y[i + 1]

        if (x1 - x0) * (x2 - x0) <= 0 and x1 != x2:
            t = (x0 - x1) / (x2 - x1)
            yi = y1 + t * (y2 - y1)

            if ylim[0] <= yi <= ylim[1]:
                ys.append(yi)

    if len(ys) == 0:
        return None

    y_center = (ylim[0] + ylim[1]) / 2
    return min(ys, key=lambda v: abs(v - y_center))


def add_graticules(ax, target_crs):
    ax.set_xticks([])
    ax.set_yticks([])

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    lon_lines = {}
    lat_lines = {}

    for lon in GRID_LONS:
        line = make_lonlat_line(lon=lon).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lon_lines[lon] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    for lat in GRID_LATS:
        line = make_lonlat_line(lat=lat).to_crs(target_crs)
        x, y = line.geometry.iloc[0].xy
        x = np.asarray(x)
        y = np.asarray(y)
        lat_lines[lat] = (x, y)

        ax.plot(
            x, y,
            color=GRID_COLOR,
            linewidth=GRID_WIDTH,
            alpha=GRID_ALPHA,
            linestyle=GRID_STYLE,
            zorder=1,
            clip_on=True
        )

    y_axis = ylim[0]
    bottom_tick_len = y_range * BOTTOM_TICK_LENGTH_RATIO
    bottom_label_offset = y_range * BOTTOM_LABEL_OFFSET_RATIO

    if ADD_BOTTOM_LABEL_AXIS_LINE:
        ax.plot(
            [xlim[0], xlim[1]],
            [y_axis, y_axis],
            color=BOTTOM_AXIS_LINE_COLOR,
            linewidth=BOTTOM_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lon in GRID_LONS:
        x_line, y_line = lon_lines[lon]
        x_label = _interp_x_at_y(x_line, y_line, y_axis, xlim)

        if x_label is None:
            continue

        if ADD_BOTTOM_LABEL_AXIS_LINE:
            ax.plot(
                [x_label, x_label],
                [y_axis, y_axis - bottom_tick_len],
                color=BOTTOM_AXIS_LINE_COLOR,
                linewidth=BOTTOM_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_label,
            y_axis - bottom_label_offset,
            f"{abs(lon):.0f}°W",
            ha="center",
            va="top",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )

    x_axis = xlim[0]
    left_tick_len = x_range * LEFT_TICK_LENGTH_RATIO
    left_label_offset = x_range * LEFT_LABEL_OFFSET_RATIO

    if ADD_LEFT_LABEL_AXIS_LINE:
        ax.plot(
            [x_axis, x_axis],
            [ylim[0], ylim[1]],
            color=LEFT_AXIS_LINE_COLOR,
            linewidth=LEFT_AXIS_LINE_WIDTH,
            zorder=20,
            clip_on=False
        )

    for lat in LAT_LABELS:
        if lat not in lat_lines:
            continue

        x_line, y_line = lat_lines[lat]
        y_label = _interp_y_at_x(x_line, y_line, x_axis, ylim)

        if y_label is None:
            continue

        if ADD_LEFT_LABEL_AXIS_LINE:
            ax.plot(
                [x_axis, x_axis - left_tick_len],
                [y_label, y_label],
                color=LEFT_AXIS_LINE_COLOR,
                linewidth=LEFT_AXIS_LINE_WIDTH,
                zorder=20,
                clip_on=False
            )

        ax.text(
            x_axis - left_label_offset,
            y_label,
            f"{lat:.0f}°N",
            ha="right",
            va="center",
            fontsize=TICK_SIZE,
            clip_on=False,
            zorder=20
        )


def add_scale_bar(ax):
    """
    添加 10 km 比例尺，并将其放在分类 legend 上方。
    位置由 SCALE_X_FRAC 和 SCALE_Y_FRAC 控制。
    """
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_range = xlim[1] - xlim[0]
    y_range = ylim[1] - ylim[0]

    x0 = xlim[0] + SCALE_X_FRAC * x_range
    y0 = ylim[0] + SCALE_Y_FRAC * y_range

    rect = Rectangle(
        (x0, y0),
        SCALE_LENGTH_M,
        SCALE_HEIGHT_M,
        facecolor="black",
        edgecolor="black",
        linewidth=0,
        zorder=45
    )
    ax.add_patch(rect)

    ax.text(
        x0 + SCALE_LENGTH_M / 2,
        y0 - SCALE_TEXT_OFFSET_M,
        SCALE_LABEL_TEXT,
        ha="center",
        va="top",
        fontsize=SCALE_TEXT_SIZE,
        color="black",
        zorder=45
    )


def get_display_xlim_ylim(outline_5070):
    """
    根据美国边界计算显示范围。

    MAP_SHRINK_FACTOR > 1 时，会扩大 xlim/ylim，
    从视觉上缩小美国地图在画布中的比例。
    """
    xmin, ymin, xmax, ymax = outline_5070.total_bounds

    x_center = (xmin + xmax) / 2
    y_center = (ymin + ymax) / 2

    raw_width = xmax - xmin
    raw_height = ymax - ymin

    width_with_pad = raw_width * (1 + 2 * PAD_X_RATIO)
    height_with_pad = raw_height * (1 + 2 * PAD_Y_RATIO)

    if FIX_WIDTH_ADJUST_HEIGHT:
        target_width = width_with_pad
        target_height = target_width * DATA_FRAME_ASPECT_RATIO
        target_height = max(target_height, height_with_pad)
    else:
        target_height = height_with_pad
        target_width = target_height / DATA_FRAME_ASPECT_RATIO
        target_width = max(target_width, width_with_pad)

    # =====================================================
    # 扩大显示范围，从而缩小美国地图在图中的占比
    # =====================================================
    target_width = target_width * MAP_SHRINK_FACTOR * MAP_SHRINK_FACTOR_X
    target_height = target_height * MAP_SHRINK_FACTOR * MAP_SHRINK_FACTOR_Y

    # =====================================================
    # 可选中心位置微调
    # =====================================================
    x_center = x_center + target_width * MAP_CENTER_SHIFT_X_RATIO
    y_center = y_center + target_height * MAP_CENTER_SHIFT_Y_RATIO

    xlim = (
        x_center - target_width / 2,
        x_center + target_width / 2
    )

    ylim = (
        y_center - target_height / 2,
        y_center + target_height / 2
    )

    return xlim, ylim


def make_point_obstacles(ax, gdf, radius_px):
    obstacles = []

    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue

        px, py = ax.transData.transform((geom.x, geom.y))

        obstacles.append(
            Bbox.from_extents(
                px - radius_px,
                py - radius_px,
                px + radius_px,
                py + radius_px
            )
        )

    return obstacles


def bbox_inside_axes(bbox, ax_bbox):
    return (
        bbox.x0 >= ax_bbox.x0 and
        bbox.x1 <= ax_bbox.x1 and
        bbox.y0 >= ax_bbox.y0 and
        bbox.y1 <= ax_bbox.y1
    )


def add_city_labels_greedy(ax, city_gdf, name_field, base_point_gdf, overlay_point_gdf):
    if name_field not in city_gdf.columns:
        raise ValueError(f"城市点数据中未找到字段：{name_field}")

    if LABEL_CLUSTER_VALUES is not None:
        label_gdf = city_gdf[
            city_gdf["cluster"].isin(LABEL_CLUSTER_VALUES)
        ].copy()
    else:
        label_gdf = city_gdf.copy()

    label_gdf = label_gdf[
        label_gdf.geometry.notnull() &
        (~label_gdf.geometry.is_empty)
    ].copy()

    label_gdf["_label_priority"] = np.where(label_gdf["cluster"] == 0, 1, 0)
    label_gdf = label_gdf.sort_values(
        by=["_label_priority", "cluster"],
        ascending=[True, True]
    )

    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ax_bbox = ax.get_window_extent(renderer)

    placed_bboxes = []

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=base_point_gdf,
            radius_px=BASE_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_bboxes.extend(
        make_point_obstacles(
            ax=ax,
            gdf=overlay_point_gdf,
            radius_px=OVERLAY_POINT_OBSTACLE_RADIUS_PX
        )
    )

    placed_count = 0
    skipped_count = 0

    for _, row in label_gdf.iterrows():
        name = row[name_field]

        if name is None or str(name).strip() == "":
            continue

        x0 = row.geometry.x
        y0 = row.geometry.y

        placed = False

        for dx, dy, ha, va in LABEL_CANDIDATE_OFFSETS:

            if CITY_LABEL_BBOX:
                bbox_style = dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=CITY_LABEL_BBOX_ALPHA,
                    pad=0.15
                )
            else:
                bbox_style = None

            txt = ax.text(
                x0 + dx,
                y0 + dy,
                str(name),
                fontsize=CITY_LABEL_SIZE,
                color=CITY_LABEL_COLOR,
                ha=ha,
                va=va,
                bbox=bbox_style,
                zorder=18
            )

            fig.canvas.draw()
            bbox = txt.get_window_extent(renderer).expanded(
                LABEL_BBOX_EXPAND_X,
                LABEL_BBOX_EXPAND_Y
            )

            if KEEP_LABELS_INSIDE_AXES and not bbox_inside_axes(bbox, ax_bbox):
                txt.remove()
                continue

            has_overlap = any(
                bbox.overlaps(old_bbox)
                for old_bbox in placed_bboxes
            )

            if has_overlap:
                txt.remove()
                continue

            placed_bboxes.append(bbox)
            placed = True
            placed_count += 1
            break

        if not placed:
            skipped_count += 1

            if not SKIP_OVERLAPPED_LABELS:
                dx, dy, ha, va = LABEL_CANDIDATE_OFFSETS[0]
                txt = ax.text(
                    x0 + dx,
                    y0 + dy,
                    str(name),
                    fontsize=CITY_LABEL_SIZE,
                    color=CITY_LABEL_COLOR,
                    ha=ha,
                    va=va,
                    zorder=18
                )
                fig.canvas.draw()
                bbox = txt.get_window_extent(renderer).expanded(
                    LABEL_BBOX_EXPAND_X,
                    LABEL_BBOX_EXPAND_Y
                )
                placed_bboxes.append(bbox)

    print(f"城市名标注完成：成功 {placed_count} 个，跳过 {skipped_count} 个。")


def add_cluster_legend(ax):
    """
    设置 cluster 与 outlier 图例：
    C1–C4 使用圆形符号；
    Outlier 使用空心菱形；
    No composite heatwave 使用灰色方块。
    """
    handles = []

    for cls in [1, 2, 3, 4]:
        style = BASE_POINT_STYLES[cls]

        handle = Line2D(
            [0],
            [0],
            marker=style["marker"],
            linestyle="None",
            label=style["label"],
            markerfacecolor=style["facecolor"],
            markeredgecolor=style["edgecolor"],
            markeredgewidth=style["linewidth"],
            markersize=LEGEND_MARKER_SIZE
        )
        handles.append(handle)

    outlier_handle = Line2D(
        [0],
        [0],
        marker="D",
        linestyle="None",
        label="Outlier reassigned by shape",
        markerfacecolor="#FFFFFF",
        markeredgecolor="#333333",
        markeredgewidth=1.1,
        markersize=LEGEND_MARKER_SIZE
    )
    handles.append(outlier_handle)

    no_heatwave_style = BASE_POINT_STYLES[0]
    no_heatwave_handle = Line2D(
        [0],
        [0],
        marker=no_heatwave_style["marker"],
        linestyle="None",
        label=no_heatwave_style["label"],
        markerfacecolor=no_heatwave_style["facecolor"],
        markeredgecolor=no_heatwave_style["edgecolor"],
        markeredgewidth=no_heatwave_style["linewidth"],
        markersize=LEGEND_MARKER_SIZE
    )
    handles.append(no_heatwave_handle)

    leg = ax.legend(
        handles=handles,
        loc=LEGEND_LOC,
        bbox_to_anchor=LEGEND_BBOX,
        fontsize=LEGEND_FONT_SIZE,
        frameon=LEGEND_FRAME,
        framealpha=LEGEND_FRAME_ALPHA,
        edgecolor=LEGEND_EDGE_COLOR,
        borderpad=0.6,
        handletextpad=0.7,
        labelspacing=0.45
    )

    leg.set_zorder(40)


# =========================================================
# 5. 读取数据
# =========================================================

outline_gdf = gpd.read_file(gdb_path, layer=outline_layer)
states_gdf = gpd.read_file(states_shp)

outline_wgs84 = outline_gdf.to_crs(epsg=4326)
states_wgs84 = states_gdf.to_crs(epsg=4326)

outline_5070 = outline_wgs84.to_crs(TARGET_CRS)
states_5070 = states_wgs84.to_crs(TARGET_CRS)

xlim, ylim = get_display_xlim_ylim(outline_5070)

city_gdf = gpd.read_file(city_cluster_path)
city_gdf = city_gdf.to_crs(TARGET_CRS)

overlay_gdf = gpd.read_file(overlay_gdb_path, layer=overlay_layer)
overlay_gdf = overlay_gdf.to_crs(TARGET_CRS)

if "cluster" not in city_gdf.columns:
    raise ValueError("圆形城市点数据中未找到字段：cluster")

if "cluster" not in overlay_gdf.columns:
    raise ValueError("棱形覆盖点数据中未找到字段：cluster")

if CITY_NAME_FIELD not in city_gdf.columns:
    raise ValueError(f"圆形城市点数据中未找到城市名字段：{CITY_NAME_FIELD}")

city_gdf["cluster"] = pd.to_numeric(
    city_gdf["cluster"],
    errors="coerce"
).astype("Int64")

overlay_gdf["cluster"] = pd.to_numeric(
    overlay_gdf["cluster"],
    errors="coerce"
).astype("Int64")

city_gdf = city_gdf.dropna(subset=["cluster"]).copy()
overlay_gdf = overlay_gdf.dropna(subset=["cluster"]).copy()

city_gdf["cluster"] = city_gdf["cluster"].astype(int)
overlay_gdf["cluster"] = overlay_gdf["cluster"].astype(int)


# =========================================================
# 6. 绘图
# =========================================================

fig, ax = plt.subplots(
    figsize=(FIG_WIDTH, FIG_HEIGHT),
    dpi=DPI,
    facecolor="white"
)

ax.set_facecolor("white")
ax.set_xlim(xlim)
ax.set_ylim(ylim)

add_graticules(ax, TARGET_CRS)

# ---------- 美国底图：浅灰色填充 ----------
states_5070.plot(
    ax=ax,
    facecolor=USA_FACE_COLOR,
    edgecolor=USA_EDGE_COLOR,
    linewidth=STATE_LINE_WIDTH,
    zorder=2
)

# ---------- 美国外边界 ----------
outline_5070.boundary.plot(
    ax=ax,
    color=OUTLINE_COLOR,
    linewidth=OUTLINE_WIDTH,
    zorder=4
)

# ---------- 圆形城市点 ----------
for cls, style in BASE_POINT_STYLES.items():
    sub = city_gdf[city_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=10
    )

# ---------- 城市名称，避让圆点和菱形 ----------
if SHOW_CITY_LABELS:
    add_city_labels_greedy(
        ax=ax,
        city_gdf=city_gdf,
        name_field=CITY_NAME_FIELD,
        base_point_gdf=city_gdf,
        overlay_point_gdf=overlay_gdf
    )

# ---------- 棱形覆盖点，最上层 ----------
for cls, style in OVERLAY_POINT_STYLES.items():
    sub = overlay_gdf[overlay_gdf["cluster"] == cls]

    if len(sub) == 0:
        continue

    ax.scatter(
        sub.geometry.x,
        sub.geometry.y,
        s=style["size"],
        marker=style["marker"],
        facecolor=style["facecolor"],
        edgecolor=style["edgecolor"],
        linewidth=style["linewidth"],
        alpha=1.0,
        zorder=25
    )

if SHOW_SCALEBAR:
    add_scale_bar(ax)

if SHOW_LEGEND:
    add_cluster_legend(ax)

for spine in ax.spines.values():
    spine.set_linewidth(SPINE_WIDTH)
    spine.set_color("black")


# =========================================================
# 7. 保存
# =========================================================

plt.subplots_adjust(
    left=0.075,
    right=0.995,
    bottom=0.08,
    top=0.99
)

if SAVE_PNG:
    out_png = out_dir / f"{output_name}.png"
    plt.savefig(
        out_png,
        dpi=DPI,
        facecolor="white"
    )
    print(f"PNG 已保存：{out_png}")

if SAVE_SVG:
    out_svg = out_dir / f"{output_name}.svg"
    plt.savefig(
        out_svg,
        format="svg",
        facecolor="white"
    )
    print(f"SVG 已保存：{out_svg}")

plt.show()
plt.close()